In [1]:
print("Federated Averaging Results")

Federated Averaging Results


## Cell 1 — Setup datasets + build client splits

In [2]:
from pathlib import Path
import sys
import torch
from torch.utils.data import DataLoader, Subset
import torch.optim as optim

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import Kits2DSegDataset
from src.transforms import TransformConfig, SegTransform
from src.models_resnet_unet import ResNet18UNet
from src.train_seg import TrainConfig, train_one_epoch, evaluate
from src.federated.partition import group_indices_by_case, iid_case_split, build_client_indices_from_cases
from src.federated.fedavg import get_state_dict, set_state_dict, fedavg

DATA_ROOT = Path(r"F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

tfm = SegTransform(TransformConfig(out_size=256, to_3ch=True, normalize_01=True))
train_ds = Kits2DSegDataset(DATA_ROOT, "train", transform=tfm)
val_ds   = Kits2DSegDataset(DATA_ROOT, "val", transform=tfm)

# Build case->indices mapping from train meta
filenames = [meta["filename"] for _, _, meta in (train_ds[i] for i in range(50))]  # quick smoke
print("Example filenames:", filenames[:3])

# full mapping (fast enough)
train_filenames = [train_ds[i][2]["filename"] for i in range(len(train_ds))]
case_to_idxs = group_indices_by_case(train_filenames)
case_ids = sorted(case_to_idxs.keys())
print("Num cases:", len(case_ids))

NUM_CLIENTS = 3
client_cases = iid_case_split(case_ids, num_clients=NUM_CLIENTS, seed=42)
client_indices = build_client_indices_from_cases(case_to_idxs, client_cases)

for c in range(NUM_CLIENTS):
    print(f"Client {c}: cases={len(client_cases[c])}, slices={len(client_indices[c])}")

Device: cuda
Example filenames: ['case_00000_axial_291.png', 'case_00000_axial_292.png', 'case_00000_axial_293.png']
Num cases: 342
Client 0: cases=114, slices=4593
Client 1: cases=114, slices=3322
Client 2: cases=114, slices=3945


## Cell 2 — Create per-client loaders

In [3]:
BATCH = 8
NUM_WORKERS = 2  # if Windows issues: 0

client_loaders = []
client_sizes = []

for c in range(NUM_CLIENTS):
    subset = Subset(train_ds, client_indices[c])
    loader = DataLoader(subset, batch_size=BATCH, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
    client_loaders.append(loader)
    client_sizes.append(len(subset))

val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))

client_sizes

[4593, 3322, 3945]

## Cell 3 — FedAvg training loop (debug run: few rounds)

In [4]:
cfg = TrainConfig(epochs=1, lr=3e-4, dice_weight=0.5, use_amp=True)  # local epochs per round
ROUNDS = 5

global_model = ResNet18UNet(num_classes=3).to(DEVICE)
global_state = get_state_dict(global_model)

history = []

for rnd in range(1, ROUNDS + 1):
    client_states = []
    client_weights = []

    for c in range(NUM_CLIENTS):
        # client model starts from global
        client_model = ResNet18UNet(num_classes=3).to(DEVICE)
        set_state_dict(client_model, global_state)

        opt = optim.AdamW(client_model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))

        # local train
        tr = train_one_epoch(client_model, client_loaders[c], opt, DEVICE, cfg, scaler=scaler)

        # collect
        client_states.append(get_state_dict(client_model))
        client_weights.append(client_sizes[c])

    # aggregate
    global_state = fedavg(client_states, client_weights)
    set_state_dict(global_model, global_state)

    # evaluate on global val
    va = evaluate(global_model, val_loader, DEVICE, cfg)
    row = {"round": rnd, **va}
    history.append(row)
    print(f"[Round {rnd}] val_loss={va['val_loss']:.4f} tumor_dice={va['tumor_dice']:.4f} tumor_iou={va['tumor_iou']:.4f}")

import pandas as pd
pd.DataFrame(history)

C:\Users\user\AppData\Local\Temp\ipykernel_9916\3181502649.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))


[Round 1] val_loss=0.6475 tumor_dice=0.0006 tumor_iou=0.0003
[Round 2] val_loss=0.4468 tumor_dice=0.2012 tumor_iou=0.1595
[Round 3] val_loss=0.3872 tumor_dice=0.3211 tumor_iou=0.2594
[Round 4] val_loss=0.3499 tumor_dice=0.3896 tumor_iou=0.3225
[Round 5] val_loss=0.4039 tumor_dice=0.2987 tumor_iou=0.2462


,round,val_loss,tumor_dice,tumor_iou,kidney_dice,kidney_iou
0,1,0.647537,0.000571,0.000314,0.054924,0.046205
1,2,0.446838,0.201181,0.159495,0.691862,0.595403
2,3,0.387169,0.321148,0.259442,0.717052,0.625040
3,4,0.349918,0.389650,0.322467,0.720207,0.629544
4,5,0.403881,0.298745,0.246218,0.726520,0.637359


## Evaluate FedAvg global model on TEST

In [5]:
from torch.utils.data import DataLoader

test_ds = Kits2DSegDataset(DATA_ROOT, "test", transform=tfm)
test_loader = DataLoader(
    test_ds, batch_size=BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda")
)

test_metrics = evaluate(global_model, test_loader, DEVICE, cfg)
print("FEDAVG TEST:", test_metrics)

FEDAVG TEST: {'val_loss': 0.46550024134897383, 'tumor_dice': 0.3119373790934363, 'tumor_iou': 0.2672242904205848, 'kidney_dice': 0.7415302704552658, 'kidney_iou': 0.6575791221921858}


In [6]:
from pathlib import Path
import torch

warm_ckpt = Path("outputs/checkpoints/resnetunet_scratch_best.pt")
ckpt = torch.load(warm_ckpt, map_location=DEVICE)
global_model.load_state_dict(ckpt["model"])
global_state = get_state_dict(global_model)

print("Warm-started FedAvg from centralized scratch checkpoint.")

C:\Users\user\AppData\Local\Temp\ipykernel_9916\1220495527.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(warm_ckpt, map_location=DEVICE)


Warm-started FedAvg from centralized scratch checkpoint.


In [7]:
global_model = ResNet18UNet(num_classes=3).to(DEVICE)
global_model.load_simclr_encoder("outputs/checkpoints/simclr_resnet18_encoder.pt")
global_state = get_state_dict(global_model)

f:\projects\hirdl\FedSSL_Paper\src\models_resnet_unet.py:95: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(simclr_encoder_ckpt, map_location="cpu")


## FedAvg with SimCLR-initialized encoder

In [8]:
from pathlib import Path

SIMCLR_ENCODER = Path("outputs/checkpoints/simclr_resnet18_encoder.pt")
assert SIMCLR_ENCODER.exists(), f"Missing: {SIMCLR_ENCODER}"

global_model = ResNet18UNet(num_classes=3).to(DEVICE)
load_info = global_model.load_simclr_encoder(str(SIMCLR_ENCODER))
print("Loaded SimCLR encoder into global model:", load_info)

global_state = get_state_dict(global_model)

Loaded SimCLR encoder into global model: {'missing': [], 'unexpected': []}


In [9]:
cfg = TrainConfig(epochs=1, lr=3e-4, dice_weight=0.5, use_amp=True)  # local epochs per round
ROUNDS = 5

global_model = ResNet18UNet(num_classes=3).to(DEVICE)
global_state = get_state_dict(global_model)

history = []

for rnd in range(1, ROUNDS + 1):
    client_states = []
    client_weights = []

    for c in range(NUM_CLIENTS):
        # client model starts from global
        client_model = ResNet18UNet(num_classes=3).to(DEVICE)
        set_state_dict(client_model, global_state)

        opt = optim.AdamW(client_model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))

        # local train
        tr = train_one_epoch(client_model, client_loaders[c], opt, DEVICE, cfg, scaler=scaler)

        # collect
        client_states.append(get_state_dict(client_model))
        client_weights.append(client_sizes[c])

    # aggregate
    global_state = fedavg(client_states, client_weights)
    set_state_dict(global_model, global_state)

    # evaluate on global val
    va = evaluate(global_model, val_loader, DEVICE, cfg)
    row = {"round": rnd, **va}
    history.append(row)
    print(f"[Round {rnd}] val_loss={va['val_loss']:.4f} tumor_dice={va['tumor_dice']:.4f} tumor_iou={va['tumor_iou']:.4f}")

import pandas as pd
pd.DataFrame(history)

C:\Users\user\AppData\Local\Temp\ipykernel_9916\3181502649.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))


[Round 1] val_loss=0.6441 tumor_dice=0.0000 tumor_iou=0.0000
[Round 2] val_loss=0.3458 tumor_dice=0.4079 tumor_iou=0.3268
[Round 3] val_loss=0.3366 tumor_dice=0.4252 tumor_iou=0.3450
[Round 4] val_loss=0.3431 tumor_dice=0.4089 tumor_iou=0.3367
[Round 5] val_loss=0.3640 tumor_dice=0.3639 tumor_iou=0.2970


,round,val_loss,tumor_dice,tumor_iou,kidney_dice,kidney_iou
0,1,0.644090,5.046736e-09,5.046736e-09,0.035035,0.035035
1,2,0.345805,4.079084e-01,3.268396e-01,0.674964,0.576610
2,3,0.336609,4.251927e-01,3.450451e-01,0.687716,0.595543
3,4,0.343088,4.088773e-01,3.366788e-01,0.724996,0.636580
4,5,0.363981,3.639044e-01,2.969701e-01,0.730414,0.641259


In [10]:
test_metrics_ssl = evaluate(global_model, test_loader, DEVICE, cfg)
print("FEDAVG + SIMCLR TEST:", test_metrics_ssl)

FEDAVG + SIMCLR TEST: {'val_loss': 0.40740443317673536, 'tumor_dice': 0.37456881392333835, 'tumor_iou': 0.314276221209238, 'kidney_dice': 0.7533532809701025, 'kidney_iou': 0.6682831152561752}


## Create a running results table (centralized vs federated)

In [15]:
import os, random
import numpy as np
import torch

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Seed fixed:", SEED)

Seed fixed: 42


In [16]:
import pandas as pd

results = [
    {"setting": "Centralized", "method": "UNet baseline", "test_tumor_dice": 0.514398313811937, "test_tumor_iou": 0.4473658952693279},
    {"setting": "Centralized", "method": "ResNetUNet scratch", "test_tumor_dice": 0.4415502547841865, "test_tumor_iou": 0.36457661939427777},
    {"setting": "Centralized", "method": "ResNetUNet + SimCLR", "test_tumor_dice": 0.5104673579206985, "test_tumor_iou": 0.42352136001645585},
    {"setting": "Federated(3c)", "method": "FedAvg supervised", "test_tumor_dice": 0.311937, "test_tumor_iou": 0.267224},
    {"setting": "Federated(3c)", "method": "FedAvg + SimCLR init", "test_tumor_dice": 0.37456881392333835, "test_tumor_iou": 0.314276221209238},
]
pd.DataFrame(results)

,setting,method,test_tumor_dice,test_tumor_iou
0,Centralized,UNet baseline,0.514398,0.447366
1,Centralized,ResNetUNet scratch,0.441550,0.364577
2,Centralized,ResNetUNet + SimCLR,0.510467,0.423521
3,Federated(3c),FedAvg supervised,0.311937,0.267224
4,Federated(3c),FedAvg + SimCLR init,0.374569,0.314276


## Save experiment results (so you never lose the true numbers)

In [13]:
import json
from pathlib import Path
from datetime import datetime

def save_run(run_name: str, cfg_dict: dict, val_history: list, test_metrics: dict):
    out = {
        "run_name": run_name,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "config": cfg_dict,
        "val_history": val_history,
        "test_metrics": test_metrics,
    }
    out_path = Path("outputs/metrics") / f"{run_name}.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(out, indent=2))
    print("Saved:", out_path.resolve())

cfg_dict = {
    "NUM_CLIENTS": NUM_CLIENTS,
    "ROUNDS": ROUNDS,
    "local_epochs_per_round": cfg.epochs,
    "batch": BATCH,
    "lr": cfg.lr,
    "dice_weight": cfg.dice_weight,
    "seed": 42,
    "init": "simclr"  # change to "random" for supervised
}

# history is your list of per-round dicts
# test_metrics_ssl is your final test dict
save_run("fedavg_simclr_3c_r5_e1", cfg_dict, history, test_metrics_ssl)

Saved: F:\projects\hirdl\FedSSL_Paper\notebooks\outputs\metrics\fedavg_simclr_3c_r5_e1.json


In [14]:
import json
import pandas as pd
from pathlib import Path

def load_metric(path):
    d = json.loads(Path(path).read_text())
    tm = d["test_metrics"]
    return {
        "setting": "Federated(3c)",
        "method": d["config"]["init"],
        "test_tumor_dice": tm["tumor_dice"],
        "test_tumor_iou": tm["tumor_iou"],
        "test_loss": tm["val_loss"],
    }

rows = []
rows.append({"setting":"Centralized","method":"UNet baseline","test_tumor_dice":0.514398313811937,"test_tumor_iou":0.4473658952693279})
rows.append({"setting":"Centralized","method":"ResNetUNet scratch","test_tumor_dice":0.4415502547841865,"test_tumor_iou":0.36457661939427777})
rows.append({"setting":"Centralized","method":"ResNetUNet + SimCLR","test_tumor_dice":0.5104673579206985,"test_tumor_iou":0.42352136001645585})

rows.append(load_metric("outputs/metrics/fedavg_supervised_3c_r5_e1.json"))   # your file name
rows.append(load_metric("outputs/metrics/fedavg_simclr_3c_r5_e1.json"))       # your file name

pd.DataFrame(rows)

FileNotFoundError: [Errno 2] No such file or directory: 'outputs\\metrics\\fedavg_supervised_3c_r5_e1.json'